In [1]:
# ============================================================
# Cell 1: Imports and Reproducibility Setup
# ============================================================

import time
import random
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import (
    resnet50, ResNet50_Weights,
    convnext_tiny, ConvNeXt_Tiny_Weights,
    efficientnet_b2, EfficientNet_B2_Weights
)

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, accuracy_score, f1_score

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print("Cell 1 : Libraries Imported")
print(f"Device : {DEVICE}")

Cell 1 : Libraries Imported
Device : cuda


## Configuration
Same hyperparameters as PneumoXNet's final training run (Notebook 09, 260px version) —
this is essential for a fair comparison. Only the model architecture changes between runs.

In [3]:
# ============================================================
# Cell 2: Configuration
# ============================================================

PROJECT_ROOT = Path("/mnt/g/Research paper/Research paper/Pneumonia-MultiModel-XAI")
DATASET_DIR = PROJECT_ROOT / "dataset" / "processed_dataset"
TRAIN_DIR = DATASET_DIR / "train"
VALID_DIR = DATASET_DIR / "validation"
TEST_DIR = DATASET_DIR / "test"

MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = 260
BATCH_SIZE = 16
EPOCHS = 20
LEARNING_RATE = 1e-4
NUM_CLASSES = 3
NUM_WORKERS = 0
CLASS_NAMES = ["BACTERIA", "NORMAL", "VIRUS"]

SEEDS = [42, 123, 2024]

RESULT_LOG_PATH = RESULTS_DIR / "baselines_multiseed_results.csv"

print(f"Epochs : {EPOCHS}  |  Seeds : {SEEDS}")

Epochs : 20  |  Seeds : [42, 123, 2024]


In [5]:
# ============================================================
# Cell 3: Transforms
# ============================================================

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(degrees=0, translate=(0.08, 0.08), scale=(0.90, 1.10)),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.10)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("Cell 3 : Transforms Ready")

Cell 3 : Transforms Ready


In [6]:
# ============================================================
# Cell 4: Model Builder Functions
# ============================================================

def build_resnet50():
    model = resnet50(weights=ResNet50_Weights.DEFAULT)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(nn.Dropout(p=0.3), nn.Linear(in_features, NUM_CLASSES))
    return model

def build_convnext_tiny():
    model = convnext_tiny(weights=ConvNeXt_Tiny_Weights.DEFAULT)
    in_features = model.classifier[2].in_features
    model.classifier[2] = nn.Linear(in_features, NUM_CLASSES)
    return model

def build_efficientnet_b2():
    model = efficientnet_b2(weights=EfficientNet_B2_Weights.DEFAULT)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(nn.Dropout(p=0.3), nn.Linear(in_features, NUM_CLASSES))
    return model

MODEL_BUILDERS = {
    "ResNet-50": build_resnet50,
    "ConvNeXt-Tiny": build_convnext_tiny,
    "EfficientNet-B2": build_efficientnet_b2
}

print("Cell 4 : Model Builders Ready ->", list(MODEL_BUILDERS.keys()))

Cell 4 : Model Builders Ready -> ['ResNet-50', 'ConvNeXt-Tiny', 'EfficientNet-B2']


In [7]:
# ============================================================
# Cell 5: Train / Validate / Test Functions
# ============================================================

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total

def validate_one_epoch(model, dataloader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return running_loss / total, correct / total

def evaluate_test_set(model, dataloader, device, class_names):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    report = classification_report(all_labels, all_preds, target_names=class_names, digits=4)
    return acc, macro_f1, report

print("Cell 5 : Functions Ready")

Cell 5 : Functions Ready


In [8]:
# ============================================================
# Cell 6: Master Loop — All Models x All Seeds (Runs Automatically)
# ============================================================

all_results = []

for model_name, builder_fn in MODEL_BUILDERS.items():
    for seed in SEEDS:

        print("=" * 70)
        print(f"Training {model_name} | Seed {seed}")
        print("=" * 70)

        set_seed(seed)

        # Fresh datasets/loaders per run (shuffle depends on seed)
        train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
        valid_dataset = datasets.ImageFolder(VALID_DIR, transform=eval_transform)
        test_dataset = datasets.ImageFolder(TEST_DIR, transform=eval_transform)

        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
        valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
        test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

        class_weights = compute_class_weight(
            class_weight="balanced", classes=np.unique(train_dataset.targets), y=train_dataset.targets
        )
        class_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

        model = builder_fn().to(DEVICE)
        criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
        optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=2e-4)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3, min_lr=1e-6)

        model_id = f"{model_name.lower().replace('-', '')}_seed{seed}"
        best_model_path = MODELS_DIR / f"{model_id}_best.pth"

        best_val_acc = 0.0
        best_epoch = 0
        patience_counter = 0
        early_stopping_patience = 7

        start_time = time.time()

        for epoch in range(EPOCHS):
            train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
            valid_loss, valid_acc = validate_one_epoch(model, valid_loader, criterion, DEVICE)
            scheduler.step(valid_acc)

            if valid_acc > best_val_acc:
                best_val_acc = valid_acc
                best_epoch = epoch + 1
                patience_counter = 0
                torch.save(model.state_dict(), best_model_path)
            else:
                patience_counter += 1

            print(f"Epoch [{epoch+1}/{EPOCHS}] Train Acc: {train_acc:.4f}  Valid Acc: {valid_acc:.4f}")

            if patience_counter >= early_stopping_patience:
                print("Early stopping triggered.")
                break

        elapsed = (time.time() - start_time) / 60

        model.load_state_dict(torch.load(best_model_path))
        test_acc, test_f1, report = evaluate_test_set(model, test_loader, DEVICE, CLASS_NAMES)

        print(f"\n{model_name} | Seed {seed} -> Test Acc: {test_acc:.4f}  Test F1: {test_f1:.4f}  Time: {elapsed:.1f} min\n")

        all_results.append({
            "model": model_name,
            "seed": seed,
            "best_epoch": best_epoch,
            "test_accuracy": test_acc,
            "test_macro_f1": test_f1,
            "training_time_min": round(elapsed, 2)
        })

        pd.DataFrame(all_results).to_csv(RESULT_LOG_PATH, index=False)   # save after every run, in case of crash

        del model
        torch.cuda.empty_cache()

print("\nALL RUNS COMPLETE")

Training ResNet-50 | Seed 42
Epoch [1/20] Train Acc: 0.7336  Valid Acc: 0.7574
Epoch [2/20] Train Acc: 0.7834  Valid Acc: 0.7836
Epoch [3/20] Train Acc: 0.8043  Valid Acc: 0.7608
Epoch [4/20] Train Acc: 0.8068  Valid Acc: 0.8087
Epoch [5/20] Train Acc: 0.8351  Valid Acc: 0.8235
Epoch [6/20] Train Acc: 0.8402  Valid Acc: 0.8041
Epoch [7/20] Train Acc: 0.8512  Valid Acc: 0.8075
Epoch [8/20] Train Acc: 0.8597  Valid Acc: 0.8223
Epoch [9/20] Train Acc: 0.8685  Valid Acc: 0.7711
Epoch [10/20] Train Acc: 0.8880  Valid Acc: 0.8269
Epoch [11/20] Train Acc: 0.8983  Valid Acc: 0.8257
Epoch [12/20] Train Acc: 0.9119  Valid Acc: 0.8371
Epoch [13/20] Train Acc: 0.9171  Valid Acc: 0.8349
Epoch [14/20] Train Acc: 0.9300  Valid Acc: 0.8178
Epoch [15/20] Train Acc: 0.9283  Valid Acc: 0.8360
Epoch [16/20] Train Acc: 0.9451  Valid Acc: 0.8166
Epoch [17/20] Train Acc: 0.9515  Valid Acc: 0.8064
Epoch [18/20] Train Acc: 0.9600  Valid Acc: 0.8405
Epoch [19/20] Train Acc: 0.9595  Valid Acc: 0.8462
Epoch [20/2